<a href="https://colab.research.google.com/github/Lawson-Dong/ESINDy_infra/blob/main/Stacking_ESINDy(another_infra).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:




class StackingSINDy(BaseSINDy):
    """Stacking Ensemble SINDy (Traditional Architecture)."""

    def __init__(self, **kwargs):
        self.n_estimators = kwargs.pop('n_estimators', 50)
        self.meta_alpha = kwargs.pop('meta_alpha', 1.0)
        self.base_models = []
        self.meta_models = []
        self.final_coefficients = None
        self.meta_learner_type = 'ridge' # 'ridge' or 'lasso'
        super().__init__(**kwargs)

    def fit(self, X, dt, verbose=False):
        n_samples, n_states = X.shape
        dX = self._compute_derivative_sg(X, dt)
        Theta = self._create_polynomial_features(X, fit=True)
        n_features = Theta.shape[1]

        # 1. 训练 K 个基模型 (在全部训练数据上)
        self.base_models = []
        for i in range(self.n_estimators):
            try:
                coeffs = []
                for j in range(n_states):
                    lasso = Lasso(alpha=self.threshold, max_iter=10000, random_state=i)
                    lasso.fit(Theta, dX[:, j])
                    coef = lasso.coef_
                    coef[np.abs(coef) < self.threshold] = 0
                    coeffs.append(coef)
                coeffs = np.array(coeffs)
                self.base_models.append(coeffs)
            except Exception:
                continue

        if len(self.base_models) == 0:
            raise RuntimeError("All base models failed.")

        # 2. 通过 KFold 生成元学习器的训练数据
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        n_base_models = len(self.base_models)

        # 存储元学习器的特征和标签
        X_meta_all = [[] for _ in range(n_states)]
        y_meta_all = [[] for _ in range(n_states)]

        for train_idx, val_idx in kf.split(X):
            Theta_train, Theta_val = Theta[train_idx], Theta[val_idx]
            dX_val = dX[val_idx]

            # 临时训练基模型 (仅限当前 Fold 的训练集)
            temp_base_preds = []
            for i in range(n_base_models):
                temp_coeffs = []
                for j in range(n_states):
                    lasso = Lasso(alpha=self.threshold, max_iter=10000, random_state=i)
                    lasso.fit(Theta_train, dX[train_idx][:, j])
                    coef = lasso.coef_
                    coef[np.abs(coef) < self.threshold] = 0
                    temp_coeffs.append(coef)
                temp_coeffs = np.array(temp_coeffs)
                pred_val = Theta_val @ temp_coeffs.T
                temp_base_preds.append(pred_val)

            # 收集当前 Fold 的元学习器数据
            n_val = len(val_idx)
            for j in range(n_states):
                X_meta_j = np.array([pred[:, j] for pred in temp_base_preds]).T  # (n_val, n_base_models)
                X_meta_all[j].append(X_meta_j)
                y_meta_all[j].append(dX_val[:, j])

        # 3. 训练元学习器并计算最终系数
        self.meta_models = []
        self.final_coefficients = np.zeros((n_states, n_features))

        for j in range(n_states):
            X_meta_j = np.vstack(X_meta_all[j])
            y_meta_j = np.concatenate(y_meta_all[j])

            if self.meta_learner_type == 'ridge':
                meta_model = Ridge(alpha=self.meta_alpha)
            elif self.meta_learner_type == 'lasso':
                meta_model = Lasso(alpha=self.meta_alpha, max_iter=10000)
            else:
                raise ValueError("Meta-learner must be 'ridge' or 'lasso'")

            meta_model.fit(X_meta_j, y_meta_j)
            self.meta_models.append(meta_model)

            alpha_weights = meta_model.coef_
            final_coeff_j = np.zeros(n_features)
            for k in range(n_base_models):
                final_coeff_j += alpha_weights[k] * self.base_models[k][j]

            self.final_coefficients[j] = final_coeff_j

        return self

    def predict_derivative(self, X):
        if X.ndim == 1:
            X = X.reshape(1, -1)
        Theta = self._create_polynomial_features(X, fit=False)
        return Theta @ self.final_coefficients.T



